# ADHD-200 fMRI site and motion robustness

Second-stage validation of the locked 409-subject A424 cohort. All nuisance regression, imputation, scaling, and classifier selection are fitted using training data only. Results are research-only and are not a clinical diagnostic system.

In [ ]:
%pip install -q scikit-learn scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.metrics import roc_auc_score

ROOT=Path('/content/drive/MyDrive/ADHD200-data')
BENCH=ROOT/'fmri/strict_loso_benchmark'
BRAINLM=ROOT/'fmri/brainlm_a424'
primary=pd.read_csv(BENCH/'locked_primary_cohort_409_with_motion.csv',dtype={'subject_id':str})
bank=np.load(BENCH/'a424_feature_bank.npz',allow_pickle=True)
assert np.array_equal(bank['subject_id'].astype(str),primary.subject_id.to_numpy())
X_fcsummary=bank['fc_summary']; X_spectral=bank['spectral']
meta=pd.read_csv(BRAINLM/'brainlm_subject_embedding_metadata.csv',dtype={'subject_id':str})
emb=np.load(BRAINLM/'brainlm_subject_embeddings.npy')
emap={s:i for i,s in enumerate(meta.subject_id.astype(str))}
X_brainlm=np.stack([emb[emap[s]] for s in primary.subject_id])
Z=primary[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(float)
print(primary.shape,X_fcsummary.shape,X_spectral.shape,X_brainlm.shape)

## 1. Training-fold nuisance residualization

For each outer and inner split, ridge regression removes variance associated with age, sex, continuous motion, scan length, and coarse QC using the training subjects only. The same fitted nuisance model is applied to the held-out subjects. This avoids leakage from the held-out site.

In [ ]:
def residualize_fold(Xtr,Ztr,Xte,Zte,nuisance_alpha=100.0):
    ix=SimpleImputer(strategy='median').fit(Xtr)
    iz=SimpleImputer(strategy='median').fit(Ztr)
    xa=ix.transform(Xtr); xb=ix.transform(Xte)
    za=iz.transform(Ztr); zb=iz.transform(Zte)
    zsc=StandardScaler().fit(za); za=zsc.transform(za); zb=zsc.transform(zb)
    nuisance=Ridge(alpha=nuisance_alpha).fit(za,xa)
    ra=xa-nuisance.predict(za); rb=xb-nuisance.predict(zb)
    xsc=StandardScaler().fit(ra)
    return xsc.transform(ra),xsc.transform(rb),za,zb

def nested_loso_residualized(X,Z,name,include_confounds=False,Cs=(0.01,0.1,1.0)):
    y=primary.label.to_numpy(); g=primary.site.astype(str).to_numpy()
    logo=LeaveOneGroupOut(); oof=np.full(len(y),np.nan); rows=[]
    for tr,te in logo.split(X,y,g):
        scores={C:[] for C in Cs}
        for itr,iva in logo.split(X[tr],y[tr],g[tr]):
            ra,rv,za,zv=residualize_fold(X[tr][itr],Z[tr][itr],X[tr][iva],Z[tr][iva])
            if include_confounds: ra=np.c_[za,ra]; rv=np.c_[zv,rv]
            for C in Cs:
                p=LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(ra,y[tr][itr]).predict_proba(rv)[:,1]
                scores[C].append(roc_auc_score(y[tr][iva],p))
        best=max(Cs,key=lambda C:np.mean(scores[C]))
        ra,rv,za,zv=residualize_fold(X[tr],Z[tr],X[te],Z[te])
        if include_confounds: ra=np.c_[za,ra]; rv=np.c_[zv,rv]
        p=LogisticRegression(C=best,class_weight='balanced',max_iter=3000).fit(ra,y[tr]).predict_proba(rv)[:,1]
        oof[te]=p
        rows.append({'model':name,'site':g[te][0],'n':len(te),'auc':roc_auc_score(y[te],p),'C':best})
    by=pd.DataFrame(rows)
    summary={'model':name,'n':len(y),'macro_auc':by.auc.mean(),
             'weighted_macro_auc':np.average(by.auc,weights=by.n),
             'pooled_oof_auc':roc_auc_score(y,oof)}
    pred=pd.DataFrame({'subject_id':primary.subject_id,'site':g,'y':y,'prob':oof,'model':name})
    return summary,by,pred

resid_sets={'brainlm_residualized':X_brainlm,
            'fc_summary_residualized':X_fcsummary,
            'spectral_residualized':X_spectral}
rs=[]; rb=[]; rp=[]
for short,X in resid_sets.items():
    for include in [False,True]:
        name=('confounds_plus_' if include else '')+short
        print('Running',name,flush=True)
        s,b,p=nested_loso_residualized(np.asarray(X,dtype=np.float32),Z,name,include_confounds=include)
        rs.append(s); rb.append(b); rp.append(p)
resid_summary=pd.DataFrame(rs).sort_values('macro_auc',ascending=False)
resid_by_site=pd.concat(rb,ignore_index=True); resid_pred=pd.concat(rp,ignore_index=True)
resid_summary.to_csv(BENCH/'residualized_imaging_summary.csv',index=False)
resid_by_site.to_csv(BENCH/'residualized_imaging_by_site.csv',index=False)
resid_pred.to_csv(BENCH/'residualized_imaging_predictions.csv',index=False)
display(resid_summary); display(resid_by_site.pivot(index='site',columns='model',values='auc'))

## 2. Site-stratified nested cross-validation

This less stringent analysis keeps every site represented in every outer fold. It tests whether the low imaging performance is caused solely by the unseen-site requirement. Folds are constructed separately within each site, while all preprocessing and hyperparameter selection remain training-only.

In [ ]:
def transform_fold(Xtr,ytr,Xte):
    imp=SimpleImputer(strategy='median').fit(Xtr)
    a=imp.transform(Xtr); b=imp.transform(Xte)
    sc=StandardScaler().fit(a)
    return sc.transform(a),sc.transform(b)

def site_stratified_fold_ids(y,g,n_splits,seed):
    fold=np.full(len(y),-1,int)
    for site in np.unique(g):
        idx=np.where(g==site)[0]
        sk=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
        for f,(_,te) in enumerate(sk.split(np.zeros(len(idx)),y[idx])):
            fold[idx[te]]=f
    assert (fold>=0).all()
    return fold

def nested_site_stratified_cv(X,name,Cs=(0.01,0.1,1.0)):
    y=primary.label.to_numpy(); g=primary.site.astype(str).to_numpy()
    outer=site_stratified_fold_ids(y,g,4,42); oof=np.full(len(y),np.nan); chosen=[]
    for f in range(4):
        tr=np.where(outer!=f)[0]; te=np.where(outer==f)[0]
        inner=site_stratified_fold_ids(y[tr],g[tr],3,100+f)
        scores={C:[] for C in Cs}
        for q in range(3):
            itr=tr[inner!=q]; iva=tr[inner==q]
            a,b=transform_fold(X[itr],y[itr],X[iva])
            for C in Cs:
                p=LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(a,y[itr]).predict_proba(b)[:,1]
                scores[C].append(roc_auc_score(y[iva],p))
        best=max(Cs,key=lambda C:np.mean(scores[C])); chosen.append(best)
        a,b=transform_fold(X[tr],y[tr],X[te])
        oof[te]=LogisticRegression(C=best,class_weight='balanced',max_iter=3000).fit(a,y[tr]).predict_proba(b)[:,1]
    rows=[]
    for site in np.unique(g):
        mask=g==site
        rows.append({'model':name,'site':site,'n':mask.sum(),'auc':roc_auc_score(y[mask],oof[mask])})
    by=pd.DataFrame(rows)
    summary={'model':name,'n':len(y),'macro_site_auc':by.auc.mean(),
             'weighted_site_auc':np.average(by.auc,weights=by.n),
             'pooled_auc':roc_auc_score(y,oof),'chosen_C':str(chosen)}
    pred=pd.DataFrame({'subject_id':primary.subject_id,'site':g,'y':y,'prob':oof,'model':name,'outer_fold':outer})
    return summary,by,pred

within_sets={'confounds':Z,'brainlm':X_brainlm,'fc_summary':X_fcsummary,
             'spectral':X_spectral,'confounds_plus_brainlm':np.c_[Z,X_brainlm]}
ws=[]; wb=[]; wp=[]
for name,X in within_sets.items():
    print('Running',name,flush=True)
    s,b,p=nested_site_stratified_cv(np.asarray(X,dtype=np.float32),'within_site__'+name)
    ws.append(s); wb.append(b); wp.append(p)
within_summary=pd.DataFrame(ws).sort_values('macro_site_auc',ascending=False)
within_by_site=pd.concat(wb,ignore_index=True); within_pred=pd.concat(wp,ignore_index=True)
within_summary.to_csv(BENCH/'within_site_cv_summary.csv',index=False)
within_by_site.to_csv(BENCH/'within_site_cv_by_site.csv',index=False)
within_pred.to_csv(BENCH/'within_site_cv_predictions.csv',index=False)
display(within_summary); display(within_by_site.pivot(index='site',columns='model',values='auc'))

## Observed result and decision

Training-fold residualization did not reveal hidden imaging signal. Residualized BrainLM obtained macro LOSO AUC 0.475; confounds plus residualized BrainLM obtained 0.605, below the original confound-only 0.686. Residualized FC summary and spectral features obtained 0.461 and 0.490.

In site-stratified nested CV, confounds obtained macro site AUC 0.663, BrainLM 0.443, spectral 0.501, FC summary 0.459, and confounds plus BrainLM 0.574. Therefore the negative imaging result is not explained solely by unseen-site evaluation. Do not fine-tune BrainLM yet.